In [ ]:
from sqlalchemy import create_engine
import urllib.parse

from faker import Faker
fake = Faker('en_US')
Faker.seed(42)

import numpy as np
np.seterr(divide = 'ignore', invalid='ignore')

import pandas as pd
pd.options.display.max_columns = None
pd.set_option('display.max_rows', 100)

import matplotlib
matplotlib.use('Agg')
from matplotlib import rcParams
%matplotlib inline
rcParams['figure.figsize'] = 19, 11

import os, random

from datetime import datetime
today = datetime.today().strftime('%Y-%m-%d')

cwd = os.getcwd()
os.makedirs('./Output', exist_ok=True) 
os.makedirs('./Input', exist_ok=True)  
os.makedirs('./model_save/model', exist_ok=True) 

fake_sk = Faker("sk_SK")
fake_us = Faker("en_US")
fakers = [fake_sk, fake_us]

Faker.seed(42)
random.seed(42)
np.random.seed(42)

address = './Input/ibm_hr_synthetic_data.csv'
df = pd.read_csv(address)

print("Original shape:", df.shape)

def generate_identity_for_gender(gender: str):
    faker = random.choice(fakers)

    # First name
    if gender == 'Male':
        first = faker.first_name_male()
    elif gender == 'Female':
        first = faker.first_name_female()
    else:
        first = faker.first_name()

    # Last name logic (SK needs gender!)
    if faker == fake_sk:
        if gender == 'Male':
            last = faker.last_name_male()
        elif gender == 'Female':
            last = faker.last_name_female()
        else:
            last = faker.last_name()
    else:
        # US priezviská sú vždy gender-neutral
        last = faker.last_name()

    full = f"{first} {last}"
    email = f"{first.lower()}.{last.lower()}@sk_company.com"
    username = f"{first[0].lower()}{last.lower()}"

    return pd.Series(
        [first, last, full, email, username],
        index=['FirstName', 'LastName', 'FullName', 'Email', 'Username']
    )

df[['FirstName', 'LastName', 'FullName', 'Email', 'Username']] = df['Gender'].apply(generate_identity_for_gender)

cat_cols = [
            'Attrition',
            'Over18',
            'BusinessTravel',
            'Gender',
            'EducationField',
            'EnvironmentSatisfaction',
            'JobInvolvement',
            'JobLevel',
            'JobRole',
            'JobSatisfaction',
            'PerformanceRating',
            'RelationshipSatisfaction',
            'WorkLifeBalance',
            'MaritalStatus',
            'OverTime',
            'Department',
            'DistanceFromHome',
            'Education',
            'FirstName', 
            'LastName', 
            'FullName', 
            'Email', 
            'Username'
            ]

cat_distributions = {
    col: df[col].value_counts(normalize=True)
    for col in cat_cols
}

numeric_cols = df.select_dtypes(include=['int64']).columns.tolist()

n_new = len(df) * 5
synthetic_rows = []

for _ in range(n_new):
    row = {}
    for col in cat_cols:
        vals = cat_distributions[col].index
        probs = cat_distributions[col].values
        row[col] = np.random.choice(vals, p=probs)
    for col in numeric_cols:
        row[col] = int(np.random.choice(df[col].values))

    gender = row.get('Gender', None)
    faker = random.choice(fakers)

    if gender == 'Male':
        first = faker.first_name_male()
    elif gender == 'Female':
        first = faker.first_name_female()
    else:
        first = faker.first_name()

    last = faker.last_name()
    full = f"{first} {last}"
    email = f"{first.lower()}.{last.lower()}@sk_company.com"
    username = f"{first[0].lower()}{last.lower()}"

    row['FirstName'] = first
    row['LastName'] = last
    row['FullName'] = full
    row['Email'] = email
    row['Username'] = username

    synthetic_rows.append(row)

df_new = pd.DataFrame(synthetic_rows)
df_new = df_new.reindex(columns=df.columns)
df_extended = pd.concat([df, df_new], ignore_index=True)


if 'EmployeeNumber' in df_extended.columns:
    df_extended['EmployeeNumber'] = range(1, len(df_extended) + 1)

for c in cat_cols:
    if c in df_extended.columns:
        df_extended[c] = df_extended[c].astype('category')

print("Extended shape:", df_extended.shape)

display(df_extended)


Original shape: (1470, 35)
Extended shape: (8820, 40)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,FirstName,LastName,FullName,Email,Username
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Y,Yes,11,3,1,80,0,8,0,1,6,4,0,5,Oľga,Ďurišová,Oľga Ďurišová,oľga.ďurišová@sk_company.com,oďurišová
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,Y,No,23,4,4,80,1,10,3,3,10,7,1,7,Aleš,Rosina,Aleš Rosina,aleš.rosina@sk_company.com,arosina
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,3,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Y,Yes,15,3,2,80,0,7,3,3,0,0,0,0,Donald,Walker,Donald Walker,donald.walker@sk_company.com,dwalker
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,4,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Y,Yes,11,3,3,80,0,8,3,3,8,7,3,0,Simona,Dudeková,Simona Dudeková,simona.dudeková@sk_company.com,sdudeková
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,5,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,Y,No,12,3,4,80,1,6,3,3,2,2,2,2,Silvester,Rosa,Silvester Rosa,silvester.rosa@sk_company.com,srosa
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8815,24,Yes,Travel_Rarely,1382,Research & Development,8,2,Medical,1,8816,3,Male,94,3,2,Laboratory Technician,3,Married,4968,5224,2,Y,No,20,3,1,80,1,3,6,2,11,4,0,3,Ignác,Gavenda,Ignác Gavenda,ignác.gavenda@sk_company.com,igavenda
8816,44,No,Travel_Rarely,1397,Research & Development,1,2,Medical,1,8817,3,Male,43,4,4,Research Scientist,3,Married,3319,10333,9,Y,No,14,3,3,80,1,24,3,3,15,0,0,13,Alexej,Huttová,Alexej Huttová,alexej.huttová@sk_company.com,ahuttová
8817,45,No,Travel_Rarely,330,Research & Development,2,1,Medical,1,8818,2,Female,79,3,3,Sales Executive,4,Single,13464,24252,1,Y,Yes,22,4,4,80,0,10,1,3,10,2,0,3,Angela,Williams,Angela Williams,angela.williams@sk_company.com,awilliams
8818,35,No,Travel_Frequently,933,Research & Development,1,3,Technical Degree,1,8819,4,Male,87,3,1,Research Director,4,Single,10748,15332,0,Y,No,19,3,2,80,1,4,3,3,5,7,0,3,Aaron,Nielsen,Aaron Nielsen,aaron.nielsen@sk_company.com,anielsen


In [2]:
df_extended.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8820 entries, 0 to 8819
Data columns (total 40 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   Age                       8820 non-null   int64   
 1   Attrition                 8820 non-null   category
 2   BusinessTravel            8820 non-null   category
 3   DailyRate                 8820 non-null   int64   
 4   Department                8820 non-null   category
 5   DistanceFromHome          8820 non-null   category
 6   Education                 8820 non-null   category
 7   EducationField            8820 non-null   category
 8   EmployeeCount             8820 non-null   int64   
 9   EmployeeNumber            8820 non-null   int64   
 10  EnvironmentSatisfaction   8820 non-null   category
 11  Gender                    8820 non-null   category
 12  HourlyRate                8820 non-null   int64   
 13  JobInvolvement            8820 non-null   catego

In [ ]:
### Save to SQL Server database table
server   = '192.168.50.88,1433'
database = 'data_science'
username = 'sa'
password = 'W4rpDr1v3@'

driver = 'ODBC Driver 17 for SQL Server'

driver_encoded = urllib.parse.quote_plus(driver)
password_encoded = urllib.parse.quote_plus(password)

connection_string = (
    f'mssql+pyodbc://{username}:{password_encoded}@{server}/{database}'
    f'?driver={driver_encoded}&Encrypt=no&TrustServerCertificate=yes'
)

print('Connecting with:', connection_string)

engine = create_engine(connection_string)
table_name = 'HR_Synth_Data'
df.to_sql(table_name, con=engine, if_exists='append', index=False, schema='dbo')
#df.info()